# EVO-1 MetaWorld Colab Runner

Run cells from top to bottom in Google Colab. The default evaluation is a small smoke test: `METAWORLD_TASK_LIMIT = 3`, `METAWORLD_EPISODES = 1`, `METAWORLD_EPISODE_HORIZON = 120`.

This notebook does not patch files under `/content/Evo-1`. Change code locally, commit and push, then let Colab reset/pull the committed code.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import subprocess

# ============================================================
# 1. Code: GitHub -> Colab /content
# ============================================================

REPO_ROOT = "/content/Evo-1"
GITHUB_URL = "https://github.com/Adamx-jin/EVO-1.git"
BRANCH = "main"

if not os.path.isdir(REPO_ROOT):
    print("Cloning code repo to /content...")
    subprocess.run(["git", "clone", "--branch", BRANCH, GITHUB_URL, REPO_ROOT], check=True)
else:
    print("Repo already exists. Resetting local Colab copy before pulling...")
    subprocess.run(["git", "-C", REPO_ROOT, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_ROOT, "reset", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only", "origin", BRANCH], check=True)

for p in [REPO_ROOT, f"{REPO_ROOT}/Evo_1", f"{REPO_ROOT}/MetaWorld_evaluation"]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Code repo ready:", REPO_ROOT)


In [ ]:
# ============================================================
# 2. Check and create persistent directories on Google Drive
# ============================================================

DATA_ROOT = "/content/drive/MyDrive/EVO-1-data"

METAWORLD_CKPT_DIR = f"{DATA_ROOT}/checkpoints/metaworld"
METAWORLD_RESULTS_DIR = f"{DATA_ROOT}/results/metaworld"
METAWORLD_LOG_DIR = f"{DATA_ROOT}/logs/metaworld"
METAWORLD_VIDEO_DIR = f"{DATA_ROOT}/episode_videos/metaworld"
METAWORLD_INSPECT_DIR = f"{DATA_ROOT}/inspect_frames/metaworld"

def ensure_dir(path):
    if os.path.exists(path):
        if os.path.isdir(path):
            print("exists :", path)
        else:
            raise RuntimeError(f"Path exists but is not a directory: {path}")
    else:
        os.makedirs(path)
        print("created:", path)

print("Checking Google Drive directories before creating anything...")
ensure_dir(DATA_ROOT)
ensure_dir(METAWORLD_CKPT_DIR)
ensure_dir(METAWORLD_RESULTS_DIR)
ensure_dir(METAWORLD_LOG_DIR)
ensure_dir(METAWORLD_VIDEO_DIR)
ensure_dir(METAWORLD_INSPECT_DIR)

print("\nChecking existing checkpoint files on Drive...")
for name in ["config.json", "norm_stats.json", "mp_rank_00_model_states.pt"]:
    path = os.path.join(METAWORLD_CKPT_DIR, name)
    print(("exists :" if os.path.exists(path) else "missing:"), path)

print("MetaWorld checkpoint dir:", METAWORLD_CKPT_DIR)
print("MetaWorld log dir:", METAWORLD_LOG_DIR)
print("MetaWorld video dir:", METAWORLD_VIDEO_DIR)

In [ ]:
# ============================================================
# 3. Install dependencies
# ============================================================

subprocess.run("pip install -q huggingface_hub", shell=True, check=True)
subprocess.run("pip uninstall -y opencv-python opencv-contrib-python", shell=True, check=False)
subprocess.run("pip install -q opencv-python-headless", shell=True, check=True)
subprocess.run("pip install -q mujoco metaworld websockets packaging psutil einops fvcore timm diffusers", shell=True, check=True)
subprocess.run('pip install -q "transformers==4.46.3" "accelerate==1.1.1"', shell=True, check=True)

print("Dependencies installed with headless OpenCV.")


In [ ]:
# ============================================================
# 4. Download MetaWorld checkpoint if missing
# ============================================================

required_ckpt_files = [
    "config.json",
    "norm_stats.json",
    "mp_rank_00_model_states.pt",
]

missing = []
for f in required_ckpt_files:
    path = os.path.join(METAWORLD_CKPT_DIR, f)
    if not os.path.exists(path):
        missing.append(f)

if missing:
    print("Missing checkpoint files:", missing)
    print("Downloading MetaWorld checkpoint from Hugging Face...")
    subprocess.run(
        [
            "hf", "download",
            "MINT-SJTU/Evo1_MetaWorld",
            "--local-dir", METAWORLD_CKPT_DIR,
        ],
        check=True,
    )
else:
    print("MetaWorld checkpoint already exists. Skip downloading.")

In [ ]:
# ============================================================
# 5. Runtime config through environment variables
# ============================================================

# Do not edit files in /content/Evo-1 here. Commit local code changes, push, then rerun Cell 1.
os.environ["EVO1_CKPT_DIR"] = METAWORLD_CKPT_DIR
EVO1_PORT = "9001"
os.environ["EVO1_SERVER_PORT"] = EVO1_PORT
os.environ["EVO1_SERVER_URL"] = f"ws://127.0.0.1:{EVO1_PORT}"
os.environ["MUJOCO_GL"] = "egl"
os.environ["QT_QPA_PLATFORM"] = "offscreen"

os.environ["METAWORLD_LOG_DIR"] = METAWORLD_LOG_DIR
os.environ["METAWORLD_VIDEO_DIR"] = METAWORLD_VIDEO_DIR
os.environ["METAWORLD_INSPECT_DIR"] = METAWORLD_INSPECT_DIR
os.environ["METAWORLD_ORDER_JSON_PATH"] = f"{REPO_ROOT}/MetaWorld_evaluation/mt50_order.json"
os.environ["METAWORLD_TASKS_JSONL_PATH"] = f"{REPO_ROOT}/MetaWorld_evaluation/tasks.jsonl"

os.environ["METAWORLD_SHOW_WINDOW"] = "0"
os.environ["METAWORLD_SAVE_VIDEO"] = "1"
os.environ["METAWORLD_SAVE_IMAGE"] = "0"

# Smoke test defaults. For full MT50, use 10, 400, and None.
os.environ["METAWORLD_EPISODES"] = "1"
os.environ["METAWORLD_EPISODE_HORIZON"] = "120"
os.environ["METAWORLD_TASK_LIMIT"] = "3"
os.environ["METAWORLD_TARGET_LEVEL"] = "all"
os.environ["METAWORLD_SEED"] = "4042"

print("Runtime config set through environment variables.")
print("EVO1_CKPT_DIR:", os.environ["EVO1_CKPT_DIR"])
print("EVO1_SERVER_URL:", os.environ["EVO1_SERVER_URL"])
print("METAWORLD_TASK_LIMIT:", os.environ["METAWORLD_TASK_LIMIT"])


In [ ]:
# ============================================================
# 6. Final file check
# ============================================================

checks = {
    "server script": f"{REPO_ROOT}/Evo_1/scripts/Evo1_server.py",
    "MetaWorld client": f"{REPO_ROOT}/MetaWorld_evaluation/mt50_evo1_client_prompt.py",
    "MetaWorld order": f"{REPO_ROOT}/MetaWorld_evaluation/mt50_order.json",
    "MetaWorld tasks": f"{REPO_ROOT}/MetaWorld_evaluation/tasks.jsonl",
    "checkpoint config": f"{METAWORLD_CKPT_DIR}/config.json",
    "norm stats": f"{METAWORLD_CKPT_DIR}/norm_stats.json",
    "model state": f"{METAWORLD_CKPT_DIR}/mp_rank_00_model_states.pt",
}

all_ok = True
for name, path in checks.items():
    ok = os.path.exists(path)
    print(("OK" if ok else "MISSING"), name, "->", path)
    if not ok:
        all_ok = False

print("\nREPO_ROOT:", REPO_ROOT)
print("METAWORLD_CKPT_DIR:", METAWORLD_CKPT_DIR)
print("RESULTS_DIR:", METAWORLD_RESULTS_DIR)
print("\nReady" if all_ok else "\nSome files are still missing.")

In [ ]:
# ============================================================
# 7. Start Evo1 websocket server
# ============================================================

import os
import signal
import subprocess
import time
import socket
import psutil
from collections import deque

SERVER_LOG = f"{METAWORLD_LOG_DIR}/evo1_server.log"
EVO1_PORT = int(os.environ.get("EVO1_SERVER_PORT", "9001"))

def port_open(host="127.0.0.1", port=EVO1_PORT):
    try:
        with socket.create_connection((host, port), timeout=2):
            return True
    except OSError:
        return False

def print_log_tail(path, n=200):
    print(f"\n===== Last {n} lines of {path} =====")
    if not os.path.exists(path):
        print("Log file does not exist yet.")
        return
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in deque(f, maxlen=n):
            print(line, end="")
    print(f"\n===== End of {path} =====")

def clear_old_evo1_server(port=EVO1_PORT):
    current_pid = os.getpid()
    killed = []

    for proc in psutil.process_iter(["pid", "name", "cmdline"]):
        try:
            pid = proc.info["pid"]
            cmdline = " ".join(proc.info.get("cmdline") or [])
            if pid != current_pid and "Evo1_server.py" in cmdline:
                proc.terminate()
                killed.append((pid, cmdline))
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

    gone, alive = psutil.wait_procs(
        [psutil.Process(pid) for pid, _ in killed if psutil.pid_exists(pid)],
        timeout=2,
    )
    for proc in alive:
        try:
            proc.kill()
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

    blockers = []
    for conn in psutil.net_connections(kind="tcp"):
        if conn.laddr and conn.laddr.port == port and conn.status == psutil.CONN_LISTEN and conn.pid:
            try:
                proc = psutil.Process(conn.pid)
                cmdline = " ".join(proc.cmdline())
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                cmdline = "<unavailable>"
            if "Evo1_server.py" in cmdline:
                try:
                    psutil.Process(conn.pid).terminate()
                    killed.append((conn.pid, cmdline))
                except (psutil.NoSuchProcess, psutil.AccessDenied):
                    pass
            else:
                blockers.append((conn.pid, cmdline))

    time.sleep(2)

    if blockers:
        print(f"Port {port} is used by a non-Evo1 process; not killing it:")
        for pid, cmdline in blockers:
            print(f"  pid={pid} cmd={cmdline}")
        raise RuntimeError(f"Port {port} is occupied by a non-Evo1 process. Change EVO1_SERVER_PORT or stop that process manually.")

    print("Old Evo1 server process cleared.")

clear_old_evo1_server(port=EVO1_PORT)

server_env = os.environ.copy()
server_env["PYTHONPATH"] = f"{REPO_ROOT}:{REPO_ROOT}/Evo_1:{REPO_ROOT}/MetaWorld_evaluation:" + server_env.get("PYTHONPATH", "")

server_proc = subprocess.Popen(
    [sys.executable, f"{REPO_ROOT}/Evo_1/scripts/Evo1_server.py"],
    cwd=f"{REPO_ROOT}/Evo_1",
    env=server_env,
    stdout=open(SERVER_LOG, "w"),
    stderr=subprocess.STDOUT,
    text=True,
)

print("Server PID:", server_proc.pid)
print("Server log:", SERVER_LOG)
print(f"Waiting for ws://127.0.0.1:{EVO1_PORT} ...")

for _ in range(600):
    if server_proc.poll() is not None:
        print_log_tail(SERVER_LOG)
        raise RuntimeError(f"Server exited early with code {server_proc.returncode}. See log above: {SERVER_LOG}")
    if port_open():
        print("Server is ready.")
        break
    time.sleep(2)
else:
    print_log_tail(SERVER_LOG)
    raise TimeoutError(f"Server did not open port {EVO1_PORT}. See log above: {SERVER_LOG}")


In [ ]:
# Optional: Show server log tail
from collections import deque

SERVER_LOG = f"{METAWORLD_LOG_DIR}/evo1_server.log"
print("Server log:", SERVER_LOG)
if os.path.exists(SERVER_LOG):
    with open(SERVER_LOG, "r", encoding="utf-8", errors="replace") as f:
        for line in deque(f, maxlen=200):
            print(line, end="")
else:
    print("No server log found yet.")


In [ ]:
# ============================================================
# 8. Run MetaWorld evaluation
# ============================================================

from collections import deque

CLIENT_LOG = f"{METAWORLD_LOG_DIR}/mt50_client_stdout.log"

def print_client_log_tail(path, n=200):
    print(f"\n===== Last {n} lines of {path} =====")
    if not os.path.exists(path):
        print("Client log file does not exist yet.")
        return
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in deque(f, maxlen=n):
            print(line, end="")
    print(f"\n===== End of {path} =====")

env = os.environ.copy()
env["MUJOCO_GL"] = "egl"
env["QT_QPA_PLATFORM"] = "offscreen"
env["PYTHONPATH"] = f"{REPO_ROOT}:{REPO_ROOT}/Evo_1:{REPO_ROOT}/MetaWorld_evaluation:" + env.get("PYTHONPATH", "")

result = subprocess.run(
    [sys.executable, "mt50_evo1_client_prompt.py"],
    cwd=f"{REPO_ROOT}/MetaWorld_evaluation",
    env=env,
    stdout=open(CLIENT_LOG, "w"),
    stderr=subprocess.STDOUT,
    text=True,
)

print("Client return code:", result.returncode)
print("Client log:", CLIENT_LOG)
print("Videos:", METAWORLD_VIDEO_DIR)

if result.returncode != 0:
    print_client_log_tail(CLIENT_LOG)
    raise RuntimeError("Evaluation failed. See client log above.")

In [ ]:
# ============================================================
# 9. Show latest evaluation log
# ============================================================

import glob

logs = sorted(glob.glob(f"{METAWORLD_LOG_DIR}/mt50_*.txt"))

print("Eval logs:")
for p in logs[-5:]:
    print(p)

if logs:
    latest = logs[-1]
    print("\nLatest log:", latest)
    print(open(latest, "r", encoding="utf-8").read()[-4000:])
else:
    print("No mt50 eval log found yet.")

In [ ]:
# Optional: stop the server when you are done.
try:
    server_proc.terminate()
    print("Server terminated.")
except NameError:
    print("server_proc is not defined in this runtime.")